# Spark create Hudi table

In [1]:
from pyspark.sql import SparkSession
import os

os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("HudiDemoNotebook") \
    .config("spark.jars", "/tmp/gravitino/spark/packages/hudi-spark3.4-bundle_2.12-1.0.2.jar") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

print("SparkSession successfully created!")

SparkSession successfully created!


In [2]:
data = [("1", "book", 10.0, "2025-09-11 09:30:00"),
        ("2", "pen", 2.0, "2025-09-11 09:31:00")]

df = spark.createDataFrame(data, ["id", "name", "price", "ts"])

hudi_options = {
    "hoodie.table.name": "hudi_table_cow",
    "hoodie.datasource.write.recordkey.field": "id",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.operation": "insert",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
}

df.write.format("hudi") \
  .option("hoodie.table.name", "hudi_table_cow") \
  .option("hoodie.datasource.write.recordkey.field", "id") \
  .option("hoodie.datasource.write.precombine.field", "ts") \
  .option("hoodie.datasource.write.operation", "upsert") \
  .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE") \
  .option("hoodie.datasource.hive_sync.enable", "true") \
  .option("hoodie.datasource.hive_sync.database", "hudi.db") \
  .option("hoodie.datasource.hive_sync.table", "hudi_table_cow") \
  .option("hoodie.datasource.hive_sync.mode", "hms") \
  .option("hoodie.datasource.hive_sync.metastore.uris", "thrift://hive:9083") \
  .mode("overwrite") \
  .save("hdfs://hive:9000/user/hive/warehouse/hudi/hudi_table_cow")

Py4JJavaError: An error occurred while calling o60.save.
: org.apache.hadoop.ipc.RemoteException(java.io.IOException): File /user/hive/warehouse/hudi/hudi_table_cow/.hoodie/hoodie.properties could only be replicated to 0 nodes instead of minReplication (=1).  There are 0 datanode(s) running and no node(s) are excluded in this operation.
	at org.apache.hadoop.hdfs.server.blockmanagement.BlockManager.chooseTarget4NewBlock(BlockManager.java:1571)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getNewBlockTargets(FSNamesystem.java:3107)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getAdditionalBlock(FSNamesystem.java:3031)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.addBlock(NameNodeRpcServer.java:725)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.addBlock(ClientNamenodeProtocolServerSideTranslatorPB.java:492)
	at org.apache.hadoop.hdfs.protocol.proto.ClientNamenodeProtocolProtos$ClientNamenodeProtocol$2.callBlockingM